# YOLOv11 Live Inference Notebook

Real-time student behavior detection from webcam using your trained weights.

Run this notebook from top to bottom.

In [ ]:
%pip install -q ultralytics opencv-python

from pathlib import Path
import time
import cv2
from ultralytics import YOLO

def is_colab() -> bool:
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False

ON_COLAB = is_colab()
print(f'Running on Colab: {ON_COLAB}')

if ON_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)

MODEL_CANDIDATES = [
    Path('/content/drive/MyDrive/fyp_runs/classroom_model_v1/weights/best.pt'),
    Path('fyp_runs/classroom_model_v1/weights/best.pt'),
]

MODEL_PATH = next((p for p in MODEL_CANDIDATES if p.exists()), MODEL_CANDIDATES[0])
CONFIDENCE_THRESHOLD = 0.5
WEBCAM_INDEX = 0
IMGSZ = 640

print(f'Model path: {MODEL_PATH}')
print(f'Confidence threshold: {CONFIDENCE_THRESHOLD}')
print(f'Webcam index: {WEBCAM_INDEX}')

## 2. Load Trained Model

This step validates and loads your `best.pt` weights.

In [ ]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f'Model not found at {MODEL_PATH}. Train first in train_model.ipynb.'
    )

print(f'Loading model from: {MODEL_PATH}')
model = YOLO(str(MODEL_PATH))
print('Model loaded successfully.')

## 3. Webcam Availability Check

Colab runtime does not support OpenCV desktop windows (`cv2.imshow`) in the normal way.
Use local Jupyter/VS Code notebook for live webcam inference.

In [ ]:
cap = cv2.VideoCapture(WEBCAM_INDEX)
if not cap.isOpened():
    print(f'Could not open webcam at index {WEBCAM_INDEX}.')
    print('Try changing WEBCAM_INDEX to 1 or 2 in Cell 2.')
else:
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f'Webcam ready: {width}x{height} @ {fps:.2f} FPS')
    ok, frame = cap.read()
    print(f'Test frame read: {ok}')
    cap.release()

## 4. Run Live Inference

Press `q` in the video window to stop.

In [ ]:
def run_live_inference() -> None:
    cap = cv2.VideoCapture(WEBCAM_INDEX)
    if not cap.isOpened():
        print(f'Could not open webcam at index {WEBCAM_INDEX}.')
        return

    print("Starting live inference. Press 'q' to stop.")
    frame_count = 0
    start_time = time.time()

    while True:
        ok, frame = cap.read()
        if not ok:
            print('Failed to read frame from webcam.')
            break

        results = model.predict(frame, conf=CONFIDENCE_THRESHOLD, imgsz=IMGSZ, verbose=False)
        annotated = results[0].plot()

        frame_count += 1
        elapsed = max(time.time() - start_time, 1e-6)
        fps = frame_count / elapsed
        det_count = len(results[0].boxes) if results[0].boxes is not None else 0

        cv2.putText(
            annotated,
            f'FPS: {fps:.1f} | Detections: {det_count}',
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2,
            cv2.LINE_AA,
        )

        cv2.imshow('Real-Time Student Behavior Detection', annotated)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print(f'Finished. Frames processed: {frame_count}')

if ON_COLAB:
    print('Colab detected: skip live webcam loop in this environment.')
    print('Use this notebook locally (VS Code/Jupyter) for webcam inference.')
else:
    run_live_inference()

## 5. Expected Behavior

- Bounding boxes and class labels appear on each detected student behavior.
- Confidence threshold is set to 0.5.
- Press `q` to quit the live window.

## Troubleshooting

- If model file is missing: run training notebook first.
- If webcam fails: change `WEBCAM_INDEX` in Cell 2 to 1 or 2.
- If FPS is low: reduce `IMGSZ` in Cell 2 from 640 to 512 or 416.